# Structured-language extraction (JSON, TOML, YAML)

**Status:** design, awaiting user review of this notebook  
**Beads epic:** `bd-2l2m`  
**Plan id:** `7c3a1e90-4b2d-4f11-9c8a-2e6f0b91d5a4`  
**Date:** 2026-08-23  
**Approved approach:** named keys only (Field / Module / Constant); lockfiles out of scope

## Solver evidence (pre-spec)

Catalog families `configuration` and `data_integrity` (Z3 4.16.0):

| Query | Status | solve_id |
|---|---|---|
| Three structured extractors + grammars, Jupyter keeps `.ipynb` | `sat` / pass | `sol_5fc0e7e624f84ff2` |
| Unique extensions, FK to languages, allowed `(family, dispatch)` | `sat` / pass | `sol_cd0123ca19864fd5` |
| Definition map Field / Module / Constant only | `sat` / pass | `sol_56a05280e1504db3` |
| JSON claims `.ipynb` | `unsat` / fail | n/a |
| Structured family + specialized dispatch | `unsat` / fail | n/a |

Same-cell NS-Mermaid gates below re-prove the implementation-facing partitions (`@spec EXT-OWNER`, `ADAPTER-PROFILE`, `NAMED-KEY`).

## Goals

- Index JSON, TOML, and YAML as first-class `spur-graph` languages so config documents (`Cargo.toml`, `package.json`, `pyproject.toml`, GitHub Actions YAML) participate in the code graph.
- Follow the existing query-driven adapter contract in `docs/spur/graph-language-adapter-contract.md` and the in-crate gate `every_registered_language_satisfies_query_contract`.
- Reuse HCL/Terraform’s *sparse* structured profile: named keys become symbols; no call graph; no new `NodeKind`.

## Non-goals

- JSONC / JSON5 / HJSON.
- Jupyter `.ipynb` (already a container language; specialized extractor in `extract/notebook.rs`).
- Terraform JSON (`.tf.json`) as Terraform addresses — those files match `json` and extract as generic JSON.
- YAML anchors/aliases as `references`.
- JSON Pointer / JSONPath edges.
- Lockfile corpora (`Cargo.lock`, `package-lock.json`, `pnpm-lock.yaml`, …).
- New ontology variants (`NodeKind::Key`, `NodeKind::Document`).
- Dual-emitting Field *and* Module for the same key.

## Gate 1 — extension owner is a total exclusive function

Every registered extension maps to exactly one owner: a structured language, Jupyter, or skip (lockfile). `.yaml` and `.yml` share YAML. `.ipynb` cannot be JSON. Lockfile basenames never enter the structured extractors even when their suffix is `json` / `toml` / `yaml`.

Formal unit: `@spec EXT-OWNER`.

In [ ]:
flowchart TD
    SPEC["`@spec EXT-OWNER
@type Ext = enum[json, toml, yaml, yml, ipynb, lock_json, lock_toml, lock_yaml]
@type Lang = enum[json, toml, yaml, jupyter, skip]
@input ext: Ext
@output status: Lang
@requires PRE: true`"]
    JSON["`@branch JSON
@when ext = json
@ensures JSON_LANG: status = json`"]
    TOML["`@branch TOML
@when ext = toml
@ensures TOML_LANG: status = toml`"]
    YAML["`@branch YAML
@when ext = yaml or ext = yml
@ensures YAML_LANG: status = yaml`"]
    NB["`@branch NOTEBOOK
@when ext = ipynb
@ensures NB_LANG: status = jupyter`"]
    SKIP["`@branch SKIP
@when ext = lock_json or ext = lock_toml or ext = lock_yaml
@ensures SKIP_LANG: status = skip`"]
    CHECK["`@verify DET: prove determinism
@verify COVER: prove partition_coverage
@verify EXCLUSIVE: prove partition_exclusive
@verify STATUSES: witness each status`"]
    SPEC --> JSON --> CHECK
    SPEC --> TOML --> CHECK
    SPEC --> YAML --> CHECK
    SPEC --> NB --> CHECK
    SPEC --> SKIP --> CHECK

## Gate 2 — adapter profile

JSON/TOML/YAML are **tree-sitter structured languages**, not Jupyter-style containers. The only legal tuples:

| family | dispatch | tags query |
|---|---|---|
| structured | tree_sitter | required (non-empty `tags.scm`) |
| notebook | specialized | empty (Jupyter only) |

Anything else (JSON-as-container, structured + empty tags) is rejected. Formal unit: `@spec ADAPTER-PROFILE`.

In [ ]:
flowchart TD
    SPEC["`@spec ADAPTER-PROFILE
@type Family = enum[structured, notebook]
@type Dispatch = enum[tree_sitter, specialized]
@type Tags = enum[required, empty]
@type Status = enum[ok, rejected]
@input family: Family
@input dispatch: Dispatch
@input tags: Tags
@output status: Status
@requires PRE: true`"]
    OK_STRUCT["`@branch OK_STRUCT
@when family = structured and dispatch = tree_sitter and tags = required
@ensures S_OK: status = ok`"]
    OK_NB["`@branch OK_NB
@when family = notebook and dispatch = specialized and tags = empty
@ensures N_OK: status = ok`"]
    BAD["`@branch BAD
@when not (family = structured and dispatch = tree_sitter and tags = required) and not (family = notebook and dispatch = specialized and tags = empty)
@ensures BAD_STATUS: status = rejected`"]
    CHECK["`@verify DET: prove determinism
@verify COVER: prove partition_coverage
@verify EXCLUSIVE: prove partition_exclusive
@verify STATUSES: witness each status`"]
    SPEC --> OK_STRUCT --> CHECK
    SPEC --> OK_NB --> CHECK
    SPEC --> BAD --> CHECK

## Gate 3 — named-key classification

Only **named** keys become symbols. Value shape decides `NodeKind`. Unnamed nodes (arrays, array elements, YAML document wrappers, alias targets without a key) are skipped.

| Value shape | Nested? | NodeKind |
|---|---|---|
| named scalar | no (top-level document child) | `Constant` |
| named scalar | yes | `Field` |
| named object / table | either | `Module` |
| array | either | skip |
| unnamed | either | skip |

A key is never both Field and Module. Nested objects inherit scope from the parent Module so FQNs are `dependencies::serde` rather than a bare colliding `serde`.

Formal unit: `@spec NAMED-KEY`.

In [ ]:
flowchart TD
    SPEC["`@spec NAMED-KEY
@type Shape = enum[scalar, object, array, unnamed]
@type Kind = enum[constant, field, module, skip]
@input shape: Shape
@input nested: Bool
@output status: Kind
@requires PRE: true`"]
    TOP_SCALAR["`@branch TOP_SCALAR
@when shape = scalar and nested = false
@ensures C_KIND: status = constant`"]
    NESTED_SCALAR["`@branch NESTED_SCALAR
@when shape = scalar and nested = true
@ensures F_KIND: status = field`"]
    OBJECT["`@branch OBJECT
@when shape = object
@ensures M_KIND: status = module`"]
    ARRAY["`@branch ARRAY
@when shape = array
@ensures A_SKIP: status = skip`"]
    UNNAMED["`@branch UNNAMED
@when shape = unnamed
@ensures U_SKIP: status = skip`"]
    CHECK["`@verify DET: prove determinism
@verify COVER: prove partition_coverage
@verify EXCLUSIVE: prove partition_exclusive
@verify STATUSES: witness each status`"]
    SPEC --> TOP_SCALAR --> CHECK
    SPEC --> NESTED_SCALAR --> CHECK
    SPEC --> OBJECT --> CHECK
    SPEC --> ARRAY --> CHECK
    SPEC --> UNNAMED --> CHECK

## Architecture

Each language is a registry row + `LanguageConfig` + `tags.scm`, matching HCL/Terraform — **not** a new extractor hook and **not** a Jupyter container.

```
file path
  -> matcher (extension + lockfile basename denylist)
  -> Language::{Json,Toml,Yaml}
  -> tree-sitter-{json,toml,yaml}
  -> queries/<lang>/tags.scm  (@definition.module|field|constant + @name)
  -> shared BytesExtractor / FactBuilder
  -> GraphFacts (contains from nesting, defines from tags)
```

JSON, TOML, and YAML **cannot** share one grammar (unlike HCL/Terraform on `tree-sitter-hcl`). They **do** share:

- definition-kind map (`definition.module` → Module, `definition.field` → Field, `definition.constant` → Constant)
- relation profile `{contains, defines}`
- lockfile basename denylist
- `builtin_method_names` → `&[]`
- no `spur-edges.scm` in v1 (no calls, imports, or address references)

`contains` is structural (definition nesting / byte-range enclosure), same as HCL without a call channel. `defines` is the tags capture set.

## Query contract

`tags.scm` follows the shared vocabulary:

- `@definition.module` on a named key whose value is an object (JSON), table (TOML), or mapping (YAML)
- `@definition.constant` on a named key whose value is a scalar and whose parent is the document root
- `@definition.field` on a named key whose value is a scalar and whose parent is a module
- inner `@name` is the **unquoted key text**

If a grammar wraps keys in quote nodes, `definition_name` may grow a small unquote arm (precedent: HCL string unquote). Do not invent language-specific field lookups when a `@name` capture will do.

Expected gate rows (`expected_definition_captures`):

```
Language::Json | Toml | Yaml => ["definition.module", "definition.field", "definition.constant"]
```

Expected relations (`expected_relation_predicates`):

```
{contains, defines}
```

Coverage matrix: `module`/`field`/`constant` = `Y`; every other definition column = `-`. Relation matrix: `contains`/`defines` = `Y`; `calls`/`imports`/`references`/`links`/notebook facts = `—`.

## Lockfile denylist

`discover_files` uses `ignore` with `standard_filters(true)`, so **gitignored** lockfiles are already skipped. Committed lockfiles (`Cargo.lock`, `package-lock.json`) are not. Exclusion is therefore a **basename denylist on the structured matchers**, not gitignore.

Minimum set (case-insensitive basename):

`Cargo.lock`, `package-lock.json`, `npm-shrinkwrap.json`, `yarn.lock`, `pnpm-lock.yaml`, `pnpm-lock.yml`, `bun.lock`, `bun.lockb`, `composer.lock`, `poetry.lock`, `uv.lock`, `Pipfile.lock`, `Gemfile.lock`, `flake.lock`, `Cargo.lock.toml` (none extra — `Cargo.lock` has no toml suffix; it is still skipped by basename).

`Cargo.lock` has extension `lock`, not `toml`, so it would not match a `.toml` matcher anyway. The denylist exists for `*.json` / `*.yaml` lockfiles whose suffix *would* match.

A matcher is: extension is in the language’s list **and** basename is not in the denylist.

## Naming / FQN

- Display label = unquoted key.
- FQN = enclosing Module chain joined with `::` (existing FactBuilder scope), e.g. `package.json` `dependencies.left-pad` → `dependencies::left-pad` if `dependencies` is a Module.
- Top-level constants keep bare names (`edition`, `name`).
- Duplicate sibling keys (YAML merge / JSON invalid) → one symbol per capture; existing edge/symbol dedup applies. No new uniqueness hook.

## Language-specific notes

**JSON**
- Document root is not a symbol (file node already exists).
- `.tf.json` becomes generic JSON (today it is skipped because no JSON language exists). Document this in `queries/README.md`; do not special-case Terraform JSON in v1.

**TOML**
- `[table]` and dotted keys (`a.b.c = 1`) become Module/Field according to NAMED-KEY.
- `[[array-of-tables]]` entries are unnamed → skip the array wrapper; named keys *inside* each table still extract (e.g. `[[bin]]` `name = "spur"` → Field `name` under whatever scope the query can attach). Do not invent `bin[0]` identities.
- Existing `ImportWorkspaceIndex` Cargo.toml crate discovery stays; dual extraction is allowed (graph keys + crate index).

**YAML**
- Multi-doc streams (`---`): skip document wrappers; extract named keys inside each document.
- Anchors/aliases: skip (no `references` in v1).
- Both `.yaml` and `.yml`.

## Schema / code change list (ordered)

TDD: failing `test(...)` then `fix`/`feat` per language family. No new `NodeKind`.

1. **`crates/spur-graph/Cargo.toml`** — add `tree-sitter-json`, `tree-sitter-toml`, `tree-sitter-yaml` compatible with workspace `tree-sitter = "0.25"`. Pin exact crates.io versions at implementation time (HCL precedent: `tree-sitter-hcl = "1.1.0"`).
2. **`extract/languages.rs`** — `Language::{Json, Toml, Yaml}`; `tree_sitter_language` / `config` / `label` / `builtin_method_names` (`&[]`) arms; shared lockfile denylist helper; three matchers; three `language_registry()` rows (`json` → `["json"]`, `toml` → `["toml"]`, `yaml` → `["yaml", "yml"]`). Uniqueness still enforced by `assert_registry_extensions_are_unique`.
3. **`queries/json/tags.scm`**, **`queries/toml/tags.scm`**, **`queries/yaml/tags.scm`** — NAMED-KEY captures only. No `spur-edges.scm` unless the query compiler rejects a missing edge query (then a comment-only file is **not** allowed; omit the queries-slice entry instead).
4. **`LanguageConfig` factories** — `definition_kind_map` with the three captures; `relation_kind_map: None`; `preserve_bare_import_path: false`; `is_method: None`; `inline_language: None`.
5. **`definition_name`** — only if `@name` still includes quotes; otherwise the default `contained_capture_text(..., "name")` path is enough.
6. **`extract/tree_sitter.rs` `language_family`** — `json` / `toml` / `yaml` identity (not folded into HCL).
7. **`store/build.rs` `MANIFEST_QUERY_BYTES`** — three `tags` entries. No schema version bump (no new NodeKind / GraphEdgeKind).
8. **Gate rows** — `expected_definition_captures`, `expected_relation_predicates`; `is_container_language` stays Jupyter-only.
9. **`queries/README.md` + `crates/spur-graph/README.md`** — coverage matrices, supported-language table (15 → 18 variants), `.tf.json` note, lockfile denylist note, Adding-A-New-Language-Family checklist.
10. **Fixtures + tests** — see Testing.

## Task decomposition (plan DAG)

| Task | Scope | Parallel? |
|---|---|---|
| T1 | Cargo.toml grammars + `Language` enum/registry/matchers/denylist + family arms | first |
| T2 | JSON `tags.scm` + config + definition query test | after T1 |
| T3 | TOML `tags.scm` + config + definition query test | after T1, // T2 |
| T4 | YAML `tags.scm` + config + definition query test | after T1, // T2 |
| T5 | Gate rows + README coverage + `MANIFEST_QUERY_BYTES` | after T2–T4 |
| T6 | Integration fixtures (package.json, Cargo.toml, action YAML, lockfile skip, `.ipynb` untouched, `.tf.json` is JSON) | after T5 |

T2/T3/T4 must not edit the same `languages.rs` regions without T1 having landed the enum/registry skeleton; query files and per-language config functions are disjoint.

## Testing

- Query compile + capture tests modeled on `crates/spur-graph/tests/hcl_definition_query.rs`.
- `every_registered_language_satisfies_query_contract` green for the three new variants.
- Fixture: `package.json` with `name` (Constant), `dependencies` (Module), nested package name (Field).
- Fixture: `Cargo.toml` `[package]` Module + `name`/`version` Fields or Constants per nesting; `[dependencies]` Module.
- Fixture: GitHub Actions YAML `jobs.build` as Module chain.
- Negative: `package-lock.json` produces **zero** facts (matcher skip, not empty parse).
- Negative: `.ipynb` still uses Jupyter container extraction, not JSON tags.
- `.tf.json` extracts JSON named keys, not `NodeKind::Resource` addresses.

## Risks

- **Grammar ABI:** tree-sitter 0.25 crates may not exist for yaml/toml under those names; implementation picks the crate that links against 0.25 and records the version in Cargo.toml.
- **Query explosion:** a naive pair-capture of every JSON object in `package-lock.json` is why the denylist is a matcher-level hard skip, not a post-parse filter.
- **TOML dotted keys / array-of-tables:** grammar node shapes differ; T3 owns that mapping, not T1.
- **YAML indentation / aliases:** malformed files already follow existing extract-error skip; aliases stay unresolved/skipped.
- **Cargo.toml dual parse:** crate index and graph keys can coexist; do not delete `ImportWorkspaceIndex` rust crate discovery.

## Formal cells that bind implementation

| `@spec` | Proofs |
|---|---|
| `EXT-OWNER` | determinism, partition_coverage, partition_exclusive, witness each status |
| `ADAPTER-PROFILE` | same |
| `NAMED-KEY` | same |

Implementation must not add a fourth `Language` that shares `json`/`toml`/`yaml`/`yml`/`ipynb`, must not mark JSON/TOML/YAML as container languages, and must not emit symbols for arrays or unnamed nodes.